In [46]:
import handcalcs.render
import numpy as np
import forallpeople as si
si.environment('structural')
import pandas as pd

# Settings

In [47]:
data_phat = r'C:\Users\jschuler\Documents\CAB_Abac_charpentes\src\abac_charpente\abac_charpente\data'

gamma_m = pd.read_csv(data_phat + r'\gamma_m.csv',sep=";",index_col='famille')
k_def = pd.read_csv(data_phat + r'\kdef.csv',sep=";")
k_mod = pd.read_csv(data_phat + r'\kmod.csv',sep=";")
limites_fleche_ec5 = pd.read_csv(data_phat + r'\limites_fleche_ec5.csv')
materiaux_bois = pd.read_csv(data_phat + r'\materiaux_bois.csv',sep=';')
params_ec5 = pd.read_csv(data_phat + r'\params_ec5.csv')
psi_ = pd.read_csv(data_phat + r'\psi.csv',sep=';')

# Données d'entrée

In [48]:
# Situation
type_poutre = "Panne" 
usage = "TOITURE_INACC" 
pente_deg = 45
classe_service = 1
classe_mecha = 'C24'
famille = 'bois_massif'
duree_charge = 'permanent'

# Charges
categorie_exploitation = "H"
g_k = 0.4 * si.kN / (si.m **2) 
# g2_k_pcent
q_k = 0.0 * si.kN / (si.m **2)
s_k = 0.36 * si.kN / (si.m **2)
w_k = 0.0 * si.kN / (si.m **2)
type_toiture_vent = "1_pan"
second_oeuvre = False 
double_flexion = True 

# Dimentions
portée = 3.5 * si.m
base = 45 * si.mm
hauteur = 120 * si.mm
entraxe = 1.2 * si.m
entraxe_antideversement = 0 * si.m
marge_securite = 0 

# Application des charges

In [ ]:
g_k = g_k * entraxe
# g2_k_pcent
q_k = q_k * entraxe
s_k = s_k * entraxe # Attention, à Projeter
w_k = w_k * entraxe
print(g_k,
      q_k,
      s_k,
      w_k)

# Combinaison ELU

In [50]:
#Coef ELU
gamma_g = 1.35  # défavorable, 1.00 = favorable
gamma_q = 1.5

psi_0,psi_1,psi_2 = psi_.loc[(psi_['cat'] == categorie_exploitation),['psi_0','psi_1','psi_2']].values[0]

In [51]:
%%render

combi_elu_str = gamma_g * g_k + gamma_q * q_k + (psi_0 * s_k + psi_0 * w_k)
combi_els_car = g_k + q_k  + (psi_0 * s_k + psi_0 * w_k)
combi_els_freq = g_k + psi_1*q_k + (psi_2 * s_k + psi_2 * w_k) 
combi_els_qperm = g_k + (psi_2 * s_k + psi_2 * w_k)

<IPython.core.display.Latex object>

# Import Carac Materieau

In [52]:
gamma_m = gamma_m.loc[famille, 'gamma_M']
k_def = k_def.loc[(k_def['famille'] == famille) & (k_def['classe_service'] == classe_service),'k_def'].values[0]
k_mod = k_mod.loc[(k_mod['famille'] == famille) & (k_mod['classe_service'] == classe_service) & (k_mod['duree_charge'] == duree_charge ),'k_mod'].values[0]
f_m_k ,f_v_k ,f_c90_k, E_0_mean = materiaux_bois.loc[(materiaux_bois['classe'] == classe_mecha),['f_m_k_MPa','f_v_k_MPa','f_c90_k_MPa','E_0_mean_MPa']].values[0] * si.MPa
rho_k = materiaux_bois.loc[(materiaux_bois['classe'] == classe_mecha),['rho_k_kgm3']].values[0] * si.kg  / si.m **2

In [53]:
%%render
f_m_d = f_m_k * k_mod / gamma_m
f_v_d = f_v_k * k_mod / gamma_m
f_c90_d = f_c90_k * k_mod / gamma_m

<IPython.core.display.Latex object>

In [54]:
%%render
Mf_z_k = (combi_elu_str * portée) / 4
I_gz = (base * hauteur **2 )/ 8
V_z = hauteur / 2
sigma_mz = Mf_z_k / (I_gz / V_z)

<IPython.core.display.Latex object>

# Taux

In [55]:
%%render
tau_flexion = sigma_mz / f_m_d
tau_fleche = 1


<IPython.core.display.Latex object>